# Notebook 05 — TF-IDF Content-Based Recommender

## Overview
Builds the content-based recommender using TF-IDF and cosine similarity.

**Output:** `Models/tfidf_matrix.npz`, `Models/vectorizer.pkl`, `Models/books_index.json`

## 1. Setup & Load

In [1]:
import json, re, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import yaml

def find_repo_root():
    current = Path().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "config.yaml").exists():
            return parent
    raise FileNotFoundError("config.yaml not found")

REPO_ROOT = find_repo_root()
with open(REPO_ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

CLEAN_DIR = REPO_ROOT / "Data" / "Clean"
MODEL_DIR = REPO_ROOT / "Models"
MODEL_DIR.mkdir(exist_ok=True)

MERGED_OUT = REPO_ROOT / config["output_data_clean_non_western_fantasy"]["merged"]
TFIDF_OUT  = REPO_ROOT / config["output_data_models_non_western_fantasy"]["tfidf_matrix"]
VEC_OUT    = REPO_ROOT / config["output_data_models_non_western_fantasy"]["vectorizer"]
INDEX_OUT  = REPO_ROOT / config["output_data_models_non_western_fantasy"]["books_index"]

with open(MERGED_OUT) as f:
    df = pd.DataFrame(json.load(f))
print(f"Loaded {len(df):,} books")

Loaded 3,396 books


## 2. Text Field Construction

Combines description, subjects, title and author. Title and author repeated x2 for extra weight.

In [2]:
def clean_title(title):
    title = re.sub(r'\s*[\(\[](hardcover|paperback|kindle edition|ebook|mass market paperback|audio cd|audiobook|unknown binding|library binding|board book|large print)[\)\]]', '', title, flags=re.IGNORECASE)
    return title.strip()

def build_text(row):
    parts = []
    if row.get("description"): parts.append(str(row["description"]))
    subjects = row.get("subjects", [])
    if isinstance(subjects, list) and subjects: parts.append(" ".join(str(s) for s in subjects))
    parts.append(str(row.get("title", "")) * 2)
    parts.append(str(row.get("author", "")) * 2)
    return " ".join(parts).strip()

df["title"] = df["title"].apply(clean_title)
df["text"]  = df.apply(build_text, axis=1)
print(f"Text field built | Avg length: {df['text'].str.len().mean():.0f} chars")

Text field built | Avg length: 991 chars


## 3. Synonym Normalization

Groups related concepts before vectorization so `witch`, `mage`, `sorcerer` are treated as the same signal.

| Group | Synonyms | Canonical |
|-------|----------|----------|
| Magic user | mage, sorcerer, wizard, witch, shaman... | `magic_user` |
| Magic | sorcery, witchcraft, enchantment... | `magic` |
| Warrior | fighter, soldier, knight, samurai... | `warrior` |
| Royalty | king, queen, emperor, empress... | `royalty` |
| Spirit/Divine | demon, god, goddess, ancestor... | `spirit` |
| Journey | quest, adventure, expedition... | `journey` |

In [5]:
SYNONYM_MAP = {
    "mage": "magic_user", "sorcerer": "magic_user", "wizard": "magic_user",
    "enchanter": "magic_user", "warlock": "magic_user", "magician": "magic_user",
    "witch": "magic_user", "shaman": "magic_user", "spellcaster": "magic_user",
    "sorcery": "magic", "witchcraft": "magic", "enchantment": "magic",
    "spells": "magic", "arcane": "magic", "mystical": "magic",
    "fighter": "warrior", "soldier": "warrior", "knight": "warrior",
    "samurai": "warrior", "hunter": "warrior", "assassin": "warrior",
    "mercenary": "warrior",
    "queen": "royalty", "king": "royalty", "emperor": "royalty",
    "empress": "royalty", "prince": "royalty", "princess": "royalty",
    "ruler": "royalty", "throne": "royalty",
    "demon": "spirit", "god": "spirit", "goddess": "spirit",
    "deity": "spirit", "ancestor": "spirit", "ghost": "spirit",
    "orisha": "spirit",
    "quest": "journey", "adventure": "journey", "expedition": "journey",
    "travel": "journey", "mission": "journey", "voyage": "journey",
}

def normalize_synonyms(text):
    words = str(text).lower().split()
    return " ".join(SYNONYM_MAP.get(w, w) for w in words)

df["text"] = df["text"].apply(normalize_synonyms)
df["subjects"] = df["subjects"].apply(lambda x: x if isinstance(x, list) else [])
df.to_json(MERGED_OUT, orient="records", indent=2, force_ascii=False)
df.to_csv(str(MERGED_OUT).replace('.json', '.csv'), index=False)
print("Synonyms normalized and dataset saved")

Synonyms normalized and dataset saved


## 4. TF-IDF Vectorization

In [6]:
custom_stop = list(ENGLISH_STOP_WORDS) + [
    "book", "novel", "fiction", "author", "story", "stories",
    "series", "readers", "bestselling", "award", "pages", "work",
    "known", "written", "writing", "writer", "read", "reading", "writers",
    "works", "collection", "life", "young", "people", "new", "world", "time", "old",
    "man", "woman", "girl", "boy", "father", "mother", "son", "daughter", "human", 
    "way", "day", "year", "years", "save", "help", "come", "comes", "find", "finds",
    "take", "takes", "make", "makes", "begin", "begins", "like", "stop", "wants", 
    "debut", "epic", "things", "left", "age", "knows", "long", "set", "just", "best",
    "na", "tan", "scott", "morgan", "american", "city", "2003", "great", "eli",
]

vectorizer = TfidfVectorizer(max_features=5000, stop_words=custom_stop, min_df=3, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df["text"])
print(f"Matrix: {tfidf_matrix.shape[0]:,} books x {tfidf_matrix.shape[1]:,} features")
print(f"Sparsity: {100 * (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.1f}%")

Matrix: 3,396 books x 5,000 features
Sparsity: 99.0%


## 5. Three-Lane Recommender

In [7]:
UNDERREPRESENTED = {
    "oceania", "australian-fantasy", "indigenous-fantasy", "indigenous_americas",
    "latin-american-fantasy", "latin_american", "south-american-fantasy",
    "middle-eastern-fantasy", "middle_eastern", "filipino", "southeast_asian",
    "african-science-fiction", "orisha", "igbo", "akan", "zulu", "yoruba",
    "anansi", "xianxia", "wuxia",
}

def recommend_three_lanes(query_title, top_n=5):
    matches = df[df["title"].str.contains(query_title, case=False, na=False)]
    if len(matches) == 0:
        print(f"'{query_title}' not found")
        return None
    idx          = matches.index[0]
    query_book   = df.iloc[idx]
    query_vec    = tfidf_matrix[idx]
    query_author = query_book["author"].lower().strip()
    print(f"Query: {query_book['title']} -- {query_book['author']}\n")
    sim_scores            = cosine_similarity(query_vec, tfidf_matrix).flatten()
    results               = df.copy()
    results["similarity"] = sim_scores
    results               = results.drop(index=idx)
    same_author = results[results["author"].str.lower().str.strip() == query_author].sort_values("similarity", ascending=False).head(top_n)
    similar     = results[results["author"].str.lower().str.strip() != query_author].sort_values("similarity", ascending=False).head(top_n)
    hidden_pool = results[(results["source_tag"].isin(UNDERREPRESENTED)) & (results["author"].str.lower().str.strip() != query_author) & (results["similarity"] >= 0.02)]
    hidden_gems = hidden_pool.sort_values("similarity", ascending=False).groupby("source_tag").first().reset_index().sort_values("similarity", ascending=False).head(top_n)
    return {"query_book": query_book, "same_author": same_author, "similar": similar, "hidden_gems": hidden_gems}

print("Recommender defined")

Recommender defined


## 6. Testing

In [8]:
cols = ["title", "author", "avg_rating", "num_ratings", "similarity", "source_tag"]
for test_title in ["Children of Blood and Bone", "The Poppy War", "Three-Body Problem"]:
    print("=" * 60)
    r = recommend_three_lanes(test_title, top_n=5)
    if r:
        print("Lane 1 - Same author:")
        print(r["same_author"][cols].to_string() if len(r["same_author"]) > 0 else "  None")
        print("\nLane 2 - Similar books:")
        print(r["similar"][cols].to_string())
        print("\nLane 3 - Hidden gems:")
        print(r["hidden_gems"][cols].to_string() if len(r["hidden_gems"]) > 0 else "  None")
    print()

Query: Children of Blood and Bone (Legacy of Orïsha, #1) -- Tomi Adeyemi

Lane 1 - Same author:
                                                      title        author  avg_rating  num_ratings  similarity       source_tag
27   Children of Anguish and Anarchy (Legacy of Orïsha, #3)  Tomi Adeyemi        3.41        18784    0.682551  african-fantasy
2   Children of Virtue and Vengeance (Legacy of Orïsha, #2)  Tomi Adeyemi        3.89        82425    0.659640  african-fantasy

Lane 2 - Similar books:
                                                title                 author  avg_rating  num_ratings  similarity       source_tag
2223                   Skin Deep Magic: Short Fiction  Craig Laurance Gidney         NaN            0    0.110476           africa
363               The Bone Witch (The Bone Witch, #1)            Rin Chupeco        3.68        51823    0.105189    asian-fantasy
63            Kingdom of Souls (Kingdom of Souls, #1)            Rena Barron        3.72         5932 

## 7. Serialize Model

In [9]:
sparse.save_npz(TFIDF_OUT, tfidf_matrix)
with open(VEC_OUT, "wb") as f:
    pickle.dump(vectorizer, f)
df.drop(columns=["text"]).to_json(INDEX_OUT, orient="records", indent=2, force_ascii=False)
print(f"Model saved to {MODEL_DIR}")
print(f"  {TFIDF_OUT.name}")
print(f"  {VEC_OUT.name}")
print(f"  {INDEX_OUT.name}")

Model saved to C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Models
  tfidf_matrix.npz
  vectorizer.pkl
  books_index.json


## Conclusions

- Pure cosine similarity works better than obscurity boost
- Synonym normalization improves recommendations
- Custom stopwords remove noise words that inflate false similarities
- Three-lane design surfaces underrepresented heritages actively

**Future improvements:** sentence embeddings, collaborative filtering, expanded synonym map